# Balabit testing first-movement pattern: user20

This notebook reads the first movement from every testing session for this user.


In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd

USER_ID = "user20"
CAPTURE_DIR_NAME = "balabit_test_files_first_movements"
PROJECT_ROOT = next(
    parent
    for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / "outputs").is_dir()
)
USER_DIR = PROJECT_ROOT / "outputs" / CAPTURE_DIR_NAME / USER_ID

session_rows = []
for json_path in sorted(USER_DIR.glob("session_*.json")):
    with json_path.open() as handle:
        capture = json.load(handle)
    for point_index, point in enumerate(capture.get("points", [])):
        session_rows.append({
            "session_id": json_path.stem,
            "point_index": point_index,
            "timestamp_ms": point["timestamp_ms"],
            "x": point["x"],
            "y": point["y"],
            "event_type": point["event_type"],
            "button": point["button"],
        })

points = pd.DataFrame(session_rows)
if points.empty:
    raise ValueError(f"No first movements found for {USER_ID} in {USER_DIR}")
print(f"User: {USER_ID}")
print(f"Sessions: {points['session_id'].nunique()}")
print(f"First-movement points: {len(points):,}")
points


In [ ]:
ordered_points = points.sort_values(
    "session_id",
    key=lambda values: values.str.replace("session_", "", regex=False).astype(int),
)

x_values = ordered_points["x"].tolist()
y_values = ordered_points["y"].tolist()
session_labels = ordered_points["session_id"].str.replace("session_", "", regex=False).tolist()

fig, ax = plt.subplots(figsize=(10, 7))
ax.plot(x_values, y_values, linewidth=1.8, color="tab:blue", alpha=0.9, label="Session order")
for x_value, y_value, session_label in zip(x_values, y_values, session_labels):
    ax.plot(x_value, y_value, marker="o", linestyle="None", markersize=7, label=f"Session {session_label}")

ax.set_title(f"Testing first-movement pattern in session order: {USER_ID}")
ax.set_xlabel("x position")
ax.set_ylabel("y position")
ax.invert_yaxis()
ax.grid(alpha=0.25)
ax.legend(title="Sessions", bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
plt.show()
